# Day 14/60 — URL Shortener (Design TinyURL / bit.ly)

## System Design Series

**Topic:** Design a URL shortening service.

| Operation | Request | Response |
|---|---|---|
| Shorten | `POST /api/v1/urls  {long_url, ttl?, alias?}` | `201 Created  {short_url, expires_at}` |
| Redirect | `GET /{short_code}` | `302 Found  Location: long_url` |

**Scale:** 100M new URLs/day · 100:1 read:write ratio · < 10ms redirect latency

> All diagrams use **Mermaid** — rendered live in Jupyter.

---

## 1. Setup: Mermaid Renderer

In [1]:
from IPython.display import HTML

def mermaid(code):
    return HTML(
        '<div class="mermaid" style="max-width:960px;background:white;'
        'padding:24px;border-radius:10px;border:1px solid #ddd;margin:12px 0;">'
        + code +
        '</div>'
        '<script type="module">'
        'import md from "https://cdn.jsdelivr.net/npm/mermaid@10/dist/mermaid.esm.min.mjs";'
        'md.initialize({startOnLoad:true,theme:"default",'
        'flowchart:{curve:"linear",padding:20},'
        'sequence:{useMaxWidth:true,mirrorActors:false}});'
        '</script>'
    )


### What this does

Defines the `mermaid()` helper used throughout this notebook. All flow diagrams are plain-text Mermaid rendered as live SVG via CDN.

## 2. System Overview: Write Path and Read Path

| Path | QPS | Latency budget | Key bottleneck |
|---|---|---|---|
| **Write** `POST /urls` | 1,157 / sec | < 500 ms | DB write + code generation |
| **Read** `GET /{code}` | 115,741 / sec | **< 10 ms** | Cache hit rate |

Reads outnumber writes **100:1**. The entire architecture is optimised for the read path. Analytics are captured **asynchronously** — they never add latency to the redirect.

In [2]:
# Cell 1 — Core problem: two APIs + 301 vs 302 redirect decision
display(mermaid("""
flowchart TD
    subgraph Write["✍️ Write Path — POST /api/v1/urls"]
        C1([Client]) -->|"POST /api/v1/urls\\nbody: {long_url, ttl?, alias?}"| API1["API Layer"]
        API1 -->|"check duplicate\\n(GSI on long_url)"| DB1[("DynamoDB\\nurl_mapping")]
        DB1 -->|"exists → return existing code"| API1
        DB1 -->|"new → generate code"| GEN["Code Generator\\n(Base62 counter)"]
        GEN -->|"store mapping"| DB1
        API1 -->|"HTTP 201 Created\\n{short_url, expires_at}"| C1
    end

    subgraph Read["🔗 Read Path — GET /{code}"]
        C2([Client]) -->|"GET /abc123"| CDN["CloudFront Edge\\n(caches 302 responses)"]
        CDN -->|"cache miss"| RL["Redirect Lambda"]
        RL -->|"GET short_code"| CACHE[("ElastiCache\\n(Valkey)")]
        CACHE -->|"miss"| DB2[("DynamoDB")]
        DB2 -->|"return long_url"| CACHE
        CACHE -->|"hit: long_url"| RL
        RL -->|"HTTP 302 Found\\nLocation: long_url"| CDN
        CDN -->|"302 → browser follows"| C2
        RL -.->|"async click event"| ANALYTICS["Kinesis\\n(analytics)"]
    end

    style C1 fill:#2E86AB,color:#fff,stroke:none
    style C2 fill:#2E86AB,color:#fff,stroke:none
    style API1 fill:#E67E22,color:#fff,stroke:none
    style RL fill:#E67E22,color:#fff,stroke:none
    style CDN fill:#27AE60,color:#fff,stroke:none
    style DB1 fill:#8E44AD,color:#fff,stroke:none
    style DB2 fill:#8E44AD,color:#fff,stroke:none
    style CACHE fill:#C44E52,color:#fff,stroke:none
    style GEN fill:#2980B9,color:#fff,stroke:none
    style ANALYTICS fill:#2E4057,color:#fff,stroke:none
"""))


### What this shows

**Write Path (top):** Client POSTs → API checks GSI for duplicate → Code Generator assigns next Base62 counter → DynamoDB PutItem → 201 Created.

**Read Path (bottom):** CloudFront edge → Redirect Lambda → ElastiCache (Valkey) → DynamoDB on miss → 302 Found. The dotted arrow to Kinesis is **async** — analytics never block the redirect response.

## 3. The Central Trade-Off: 301 vs 302 Redirect

| | 301 Permanent | 302 Temporary |
|---|---|---|
| Browser caches? | **Yes** — indefinitely | **No** — re-requests every time |
| Analytics accuracy | Broken after visit 1 | **100% accurate** |
| Can update destination? | No | **Yes** |
| Used by | TinyURL | **bit.ly, t.co** |

> **Production default: 302.** Analytics is the core product. ElastiCache (not the browser) absorbs the repeat traffic.

In [3]:
# Cell 2 — 301 vs 302: the most misunderstood decision in URL shortener design
display(mermaid("""
flowchart LR
    subgraph R301["301 Permanent Redirect"]
        UA1([Browser]) -->|"GET /abc123 (visit 1)"| S1["Server"]
        S1 -->|"301 Location: long_url\\nCache-Control: max-age=31536000"| UA1
        UA1 -->|"visit 2, 3, 4...\\nGO DIRECT — server never sees it"| DEST1["Destination"]
    end

    subgraph R302["302 Temporary Redirect  ← bit.ly / production default"]
        UA2([Browser]) -->|"GET /abc123 (visit 1)"| S2["Server"]
        S2 -->|"302 Location: long_url\\nno cache directive"| UA2
        UA2 -->|"visit 2: GET /abc123 again"| S2
        UA2 -->|"visit 3: GET /abc123 again"| S2
        S2 -->|"every request logged ✅"| LOG["Analytics\\nStore"]
    end

    style UA1 fill:#2E86AB,color:#fff,stroke:none
    style UA2 fill:#2E86AB,color:#fff,stroke:none
    style S1 fill:#C44E52,color:#fff,stroke:none
    style S2 fill:#27AE60,color:#fff,stroke:none
    style DEST1 fill:#8E44AD,color:#fff,stroke:none
    style LOG fill:#E67E22,color:#fff,stroke:none
"""))


### What this shows

**302 (right):** Every visit hits the Shortener Server. All three clicks logged.

**301 (left):** Visit 1 hits the server; browser caches the response. Visits 2 and 3 bypass the server entirely — clicks never logged.

## 4. Base62 Encoding: Counter → 7-Character Short Code

Character set: `[0-9A-Za-z]` — 62 URL-safe characters, no percent-encoding needed.

| Length | Capacity | At 1,157 writes/sec, exhausted in |
|---|---|---|
| 6 | 62^6 = 56 billion | ~1.6 years |
| **7** | **62^7 = 3.5 trillion** | **~96 years** |

**Counter-based algorithm:** issue next integer (Redis `INCR`) → integer-divide by 62 → look up each remainder in char table → reverse → left-pad to 7. No collision check needed — each integer is globally unique.

In [4]:
# Cell 3 -- Base62 Base62: how 7 characters give 3.5 trillion unique URLs
print("=" * 62)
print("BASE62 ENCODING: CHARACTER SET AND ALGORITHM")
print("=" * 62)

CHARS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz"
BASE  = 62

def encode(n, length=7):
    if n == 0:
        return CHARS[0] * length
    result = []
    while n:
        result.append(CHARS[n % BASE])
        n //= BASE
    # left-pad to `length` with '0'
    while len(result) < length:
        result.append(CHARS[0])
    return ''.join(reversed(result))

# Capacity
capacity = BASE ** 7
print(f"\nCharacter set : {CHARS}")
print(f"Base          : {BASE}")
print(f"Code length   : 7 characters")
print(f"\n62^7          = {capacity:,}  (~3.5 trillion unique codes)")
print(f"\nAt 100M URLs/day → {100_000_000 / capacity * 100:.4f}% exhausted per day")
print(f"Years to exhaust: {capacity / 100_000_000 / 365:.1f} years\n")

print("-" * 62)
print("ENCODING EXAMPLES (counter → Base62 short code)")
print("-" * 62)
examples = [0, 1, 61, 62, 100_000_000, 999_999_999, capacity - 1]
for n in examples:
    code = encode(n)
    print(f"  counter {n:>20,}  →  {code}")

print()
print("=" * 62)
print("HASH-BASED vs COUNTER-BASED: COMPARISON")
print("=" * 62)
comparison = [
    ("Uniqueness",   "Guaranteed (global counter)", "Collision possible (must check DB)"),
    ("Predictability","Sequential -- enumerable!",  "Random -- unpredictable ✅"),
    ("Coordination", "Needs distributed counter",  "No coordination needed"),
    ("Speed",        "One DB write",                "One DB read (collision check)"),
    ("Recommendation","Use with counter pre-generation","Use MD5/SHA256, take first 7 chars"),
]
print(f"\n{'Dimension':<20} {'Counter-Based':<35} {'Hash-Based':<35}")
print("-" * 90)
for dim, counter, hashed in comparison:
    print(f"{dim:<20} {counter:<35} {hashed:<35}")


BASE62 ENCODING: CHARACTER SET AND ALGORITHM

Character set : 0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
Base          : 62
Code length   : 7 characters

62^7          = 3,521,614,606,208  (~3.5 trillion unique codes)

At 100M URLs/day → 0.0028% exhausted per day
Years to exhaust: 96.5 years

--------------------------------------------------------------
ENCODING EXAMPLES (counter → Base62 short code)
--------------------------------------------------------------
  counter                    0  →  0000000
  counter                    1  →  0000001
  counter                   61  →  000000z
  counter                   62  →  0000010
  counter          100,000,000  →  006laZE
  counter          999,999,999  →  015ftgF
  counter    3,521,614,606,207  →  zzzzzzz

HASH-BASED vs COUNTER-BASED: COMPARISON

Dimension            Counter-Based                       Hash-Based                         
----------------------------------------------------------------------------

### What this shows

Live Python Base62 encoder demonstrating:
- The 62-char set and the integer-division loop
- `62^7 = 3,521,614,606,208` — ~96 years at 100M URLs/day
- Examples: counter `1,000,000` → `0004C92`; max counter → `zzzzzzz`
- Comparison: counter-based (no collision, sequential) vs hash-based (random, needs retry)

## 5. Scale Estimation

**Assumptions:** 100M URLs/day · 100:1 read:write · 500 bytes/record · 10-year retention

```
Writes/sec      : 100M / 86,400     = 1,157 QPS
Reads/sec       : 1,157 × 100       = 115,741 QPS
Storage 10yr    : 365B records × 500B ≈ 182 TB
Cache (top 20%) : 20M records × 500B ≈ 9.3 GB
```

In [5]:
# Cell 4 — Scale estimation: writes/sec, reads/sec, storage, cache sizing
print("=" * 62)
print("SCALE ESTIMATION (100M URLs created / day)")
print("=" * 62)

# Write side
daily_writes  = 100_000_000
write_qps     = daily_writes / 86_400
READ_RATIO    = 100
read_qps      = write_qps * READ_RATIO

print(f"\n{'WRITE SIDE':}")
print(f"  Daily new URLs        : {daily_writes:,}")
print(f"  Writes per second     : {write_qps:,.1f} QPS")

print(f"\n{'READ SIDE (100:1 read:write ratio):':}")
print(f"  Reads per second      : {read_qps:,.0f} QPS")
print(f"  Peak (3× average)     : {read_qps * 3:,.0f} QPS")

# Storage
print(f"\n{'STORAGE (10-year horizon):':}")
record_bytes  = 500          # short_code 7B + long_url ~200B + metadata ~293B
years         = 10
total_records = daily_writes * 365 * years
total_gb      = total_records * record_bytes / 1e9
total_tb      = total_gb / 1000
print(f"  Records per year      : {daily_writes * 365:,}")
print(f"  Records (10 years)    : {total_records:,}")
print(f"  Bytes per record      : {record_bytes} B")
print(f"  Total storage         : {total_gb:,.0f} GB  ≈  {total_tb:.1f} TB")
print(f"  DynamoDB note         : auto-scales; ~$0.25/GB-month on-demand")

# Cache sizing (80/20 rule)
print(f"\n{'CACHE SIZING (80/20 rule):':}")
hot_pct       = 0.20         # 20% of URLs get 80% of traffic
cache_records = daily_writes * hot_pct
cache_bytes   = cache_records * record_bytes
cache_gb      = cache_bytes / 1e9
print(f"  Hot URLs (top 20%)    : {cache_records:,.0f} records")
print(f"  Cache memory needed   : {cache_gb:.1f} GB")
print(f"  ElastiCache node      : cache.r7g.large (12.93 GB) — covers hot tier")
print(f"  Expected cache hit    : ~80% (long-tail URLs go to DynamoDB)")

print(f"\n{'BANDWIDTH:':}")
avg_url_len   = 200          # bytes
redirect_bw   = read_qps * (7 + avg_url_len)   # short code + long URL in Location header
print(f"  Redirect responses    : {redirect_bw / 1e6:.1f} MB/s")
print(f"  CloudFront offload    : ~95% → origin sees ~{redirect_bw * 0.05 / 1e6:.1f} MB/s")

print()
print("=" * 62)
print("SUMMARY")
print("=" * 62)
summary = [
    ("Writes",    f"{write_qps:,.0f} QPS"),
    ("Reads",     f"{read_qps:,.0f} QPS   (100:1 ratio)"),
    ("Peak reads",f"{read_qps*3:,.0f} QPS   (3× burst)"),
    ("Storage",   f"{total_tb:.1f} TB over 10 years"),
    ("Cache",     f"{cache_gb:.1f} GB for 80% hit rate"),
]
for k, v in summary:
    print(f"  {k:<14} {v}")


SCALE ESTIMATION (100M URLs created / day)

WRITE SIDE
  Daily new URLs        : 100,000,000
  Writes per second     : 1,157.4 QPS

READ SIDE (100:1 read:write ratio):
  Reads per second      : 115,741 QPS
  Peak (3× average)     : 347,222 QPS

STORAGE (10-year horizon):
  Records per year      : 36,500,000,000
  Records (10 years)    : 365,000,000,000
  Bytes per record      : 500 B
  Total storage         : 182,500 GB  ≈  182.5 TB
  DynamoDB note         : auto-scales; ~$0.25/GB-month on-demand

CACHE SIZING (80/20 rule):
  Hot URLs (top 20%)    : 20,000,000 records
  Cache memory needed   : 10.0 GB
  ElastiCache node      : cache.r7g.large (12.93 GB) — covers hot tier
  Expected cache hit    : ~80% (long-tail URLs go to DynamoDB)

BANDWIDTH:
  Redirect responses    : 24.0 MB/s
  CloudFront offload    : ~95% → origin sees ~1.2 MB/s

SUMMARY
  Writes         1,157 QPS
  Reads          115,741 QPS   (100:1 ratio)
  Peak reads     347,222 QPS   (3× burst)
  Storage        182.5 TB over 

### What this shows

Precise calculations for all four sizing dimensions with Python arithmetic.

Key insight: the 100:1 read:write ratio means the **cache hit rate directly determines cost** — every 1% improvement saves ~1,157 DynamoDB reads/second.

## 6. Database Design: Why DynamoDB

Hot path: `GetItem(PK='Ab3xY9z')` — single-key lookup, no JOINs. Textbook DynamoDB.

| | SQL | DynamoDB |
|---|---|---|
| Access pattern | Flexible | Key-value only needed |
| Write scaling | Single primary | Unlimited, auto-partitioned |
| URL expiry | Cron cleanup job | **Native TTL — free, automatic** |
| Connection limits | ~500 max | None |

**TTL:** Set `expires_at` = Unix epoch. DynamoDB auto-deletes within ~48h at zero cost. Deletes stream to Lambda → invalidates ElastiCache key.

In [6]:
# Cell 5 — Database design: DynamoDB schema + access patterns + TTL
display(mermaid("""
flowchart TD
    subgraph Schema["DynamoDB Table: url_mapping"]
        PK["🔑 Partition Key: short_code  (String, 7 chars)\\ne.g.  'Ab3xY9z'"]
        ATTRS["Attributes:\\n• long_url       String   'https://example.com/very/long/path'\\n• user_id        String   'user_8f3a...'\\n• created_at     Number   1720000000  (Unix epoch)\\n• expires_at     Number   1751536000  (TTL attribute — DynamoDB auto-deletes)\\n• click_count    Number   4821\\n• is_custom      Bool     false"]
    end

    subgraph GSI1["GSI 1: reverse_lookup_index"]
        G1PK["Partition Key: long_url"]
        G1NOTE["Purpose: check if long_url already shortened\\n(avoid duplicates on POST)"]
    end

    subgraph GSI2["GSI 2: user_history_index"]
        G2PK["Partition Key: user_id"]
        G2SK["Sort Key: created_at"]
        G2NOTE["Purpose: 'list all my short URLs'\\nPaginated, sorted newest-first"]
    end

    subgraph TTL["TTL: How Automatic Expiry Works"]
        T1["expires_at attribute set at creation\\n(e.g. now + 30 days)"]
        T2["DynamoDB TTL scanner runs in background"]
        T3["Expired items deleted within ~48 hours\\nNo extra cost — free feature"]
        T4["Deleted items appear in DynamoDB Streams\\n→ trigger Lambda to invalidate ElastiCache"]
        T1 --> T2 --> T3 --> T4
    end

    subgraph Access["Access Pattern Analysis"]
        AP1["GET /{code}\\n→ DynamoDB GetItem(short_code)\\n→ O(1) single partition key lookup ✅"]
        AP2["POST /urls (duplicate check)\\n→ Query GSI1(long_url)\\n→ Strong consistent read on GSI"]
        AP3["GET /user/history\\n→ Query GSI2(user_id, sort by created_at)\\n→ Paginated with ExclusiveStartKey"]
    end

    style PK fill:#8E44AD,color:#fff,stroke:none
    style G1PK fill:#2980B9,color:#fff,stroke:none
    style G2PK fill:#2980B9,color:#fff,stroke:none
    style T1 fill:#27AE60,color:#fff,stroke:none
    style T3 fill:#27AE60,color:#fff,stroke:none
"""))


### What this shows

Four subgraphs: Schema (7 attributes, `short_code` as PK) · GSI 1 (reverse lookup by `long_url`) · GSI 2 (user history by `user_id`) · TTL flow (expires_at → auto-delete → DynamoDB Streams → ElastiCache invalidation).

## 7. Cache Design: Cache-Aside with LRU + Negative Caching

Latency budget:

| Layer | Latency |
|---|---|
| ElastiCache HIT | **~0.15 ms** |
| DynamoDB GetItem | ~3 ms |

**Cache keys:**
```
url:{short_code}            TTL 86400s — found URL
url:NOTFOUND:{short_code}   TTL 300s   — negative caching (prevents DDB hammering)
```

**Eviction:** `allkeys-lru` — viral links are accessed right after being shared, then drop off. LRU correctly keeps recent hot links and evicts stale ones.

In [7]:
# Cell 6 — Cache design: read path + LRU eviction trace
display(mermaid("""
flowchart TD
    subgraph ReadPath["Read Path with Cache (115,000 QPS)"]
        REQ["Incoming GET /{code}"] --> CDN["CloudFront Edge\\n~95% hits served here\\nfrom edge PoP"]
        CDN -->|"cache miss (5%)"| LAMBDA["Redirect Lambda"]
        LAMBDA -->|"GET url:{code}"| EC["ElastiCache (Valkey)\\n~0.15ms lookup"]
        EC -->|"HIT (80%) → long_url"| LAMBDA
        EC -->|"MISS (20%)"| DDB["DynamoDB\\n~3ms lookup"]
        DDB -->|"SET url:{code} ttl=86400"| EC
        DDB -->|"long_url"| LAMBDA
        LAMBDA -->|"302 Location: long_url"| CDN
        LAMBDA -.->|"async\\nclick event"| KIN["Kinesis\\nData Streams"]
    end

    subgraph CacheKeys["Cache Key Design"]
        K1["url:{short_code}     TTL: 24h\\nValue: long_url string"]
        K2["url:NOTFOUND:{code}  TTL: 5min\\nValue: '404' sentinel\\n(negative caching — prevents DDB hammering)"]
    end

    subgraph LRU["LRU Eviction: What Happens at 12.93 GB Capacity"]
        direction LR
        LRU_S["Cache Full\\n[MRU] Z9p... → Ab3... → xK7... → Qq1... [LRU]"]
        HIT["Request: GET Ab3xY9z\\n→ CACHE HIT"]
        PROMOTE["Promote Ab3... to MRU\\n[Ab3...] → Z9p... → xK7... → Qq1..."]
        NEW["Request: GET NEW_CODE\\n→ CACHE MISS → fetch DDB"]
        EVICT["Evict LRU: Qq1...\\n[NEW...] → Ab3... → Z9p... → xK7..."]
        LRU_S --> HIT --> PROMOTE --> NEW --> EVICT
    end

    style CDN fill:#27AE60,color:#fff,stroke:none
    style EC fill:#C44E52,color:#fff,stroke:none
    style DDB fill:#8E44AD,color:#fff,stroke:none
    style LAMBDA fill:#E67E22,color:#fff,stroke:none
    style KIN fill:#2E4057,color:#fff,stroke:none
    style K2 fill:#C44E52,color:#fff,stroke:none
"""))


### What this shows

**Read Path:** CloudFront edge (~95% served) → Lambda → ElastiCache (80% hit) → DynamoDB (20% miss). Analytics: async dotted arrow, never blocks 302.

**Cache Keys:** Positive (24h) and negative (5min) key patterns.

**LRU Trace:** Viral link `Ab3xY9z` promoted to MRU on access. New entry evicts the least-recently-used slot.

## 8. Complete AWS Production Architecture

In [8]:
# Cell 7 — Complete AWS production architecture
display(mermaid("""
flowchart TD
    subgraph Internet["Client Traffic"]
        CL([Browser / App])
    end

    subgraph Edge["Edge Layer"]
        CF["Amazon CloudFront\\n(cache 302 redirects at PoP)"]
        WAF["AWS WAF\\n(rate-based rules: 100 req/IP/min)"]
        CL -->|"HTTPS"| WAF --> CF
    end

    subgraph API["API / Compute Layer"]
        APIGW["API Gateway (HTTP API)\\nUsage Plans: 1000 req/day free tier"]
        RL["Lambda: Redirect\\n(GET /{code} → 302)"]
        SL["Lambda: Shortener\\n(POST /api/v1/urls → 201)"]
        CF -->|"GET /{code}"| RL
        CF -->|"POST /api/v1/urls"| APIGW --> SL
    end

    subgraph Cache["Cache Layer"]
        EC["ElastiCache (Valkey)\\nMulti-AZ  •  LRU eviction\\n~80% hit rate"]
        RL <-->|"GET url:{code}"| EC
        SL <-->|"SET url:{code}"| EC
    end

    subgraph DB["Storage Layer"]
        DDB["DynamoDB\\nOn-demand capacity\\nTTL enabled\\nGSI: long_url, user_id"]
        EC -->|"miss"| DDB
        SL -->|"PutItem"| DDB
    end

    subgraph Analytics["Analytics Pipeline (async)"]
        KIN["Kinesis Data Streams\\n(click events)"]
        FH["Kinesis Firehose\\n(buffer + batch)"]
        S3A["Amazon S3\\n(raw click logs)"]
        ATH["Amazon Athena\\n(SQL queries on S3)"]
        RL -.->|"async event"| KIN --> FH --> S3A --> ATH
    end

    subgraph Ops["Operations"]
        CW["CloudWatch Alarms\\nLatency • Error rate • Cache hit"]
        XRAY["AWS X-Ray\\n(trace: CDN→Lambda→Cache→DDB)"]
    end

    style CL fill:#2E86AB,color:#fff,stroke:none
    style WAF fill:#C44E52,color:#fff,stroke:none
    style CF fill:#27AE60,color:#fff,stroke:none
    style APIGW fill:#E67E22,color:#fff,stroke:none
    style RL fill:#E67E22,color:#fff,stroke:none
    style SL fill:#E67E22,color:#fff,stroke:none
    style EC fill:#C44E52,color:#fff,stroke:none
    style DDB fill:#8E44AD,color:#fff,stroke:none
    style KIN fill:#2E4057,color:#fff,stroke:none
    style FH fill:#2E4057,color:#fff,stroke:none
    style S3A fill:#2980B9,color:#fff,stroke:none
    style ATH fill:#2980B9,color:#fff,stroke:none
"""))


### What this shows

Six-tier production architecture:

1. **Edge:** WAF rate rules → CloudFront (cache redirects at global PoPs)
2. **API:** API Gateway Usage Plans (per-client throttle)
3. **Compute:** Redirect Lambda (provisioned concurrency) + Shortener Lambda
4. **Cache:** ElastiCache Valkey Multi-AZ — 95%+ hit rate
5. **DB:** DynamoDB on-demand, TTL, Streams → cache invalidation Lambda
6. **Analytics:** Kinesis → Firehose → S3 → Athena (fully async, zero latency impact)

**Three key choices:** Lambda (spiky viral traffic) · 302 (analytics) · DynamoDB (single-key lookups)

## AWS Production Notes

> Based on the AWS Well-Architected Framework.

### Top 3 Interview Mistakes

| Mistake | Why Wrong | Correct |
|---|---|---|
| 301 when analytics needed | Browser caches → server never sees repeat visits | 302: every click hits server |
| SQL as primary store | 115K QPS single-key lookup exhausts connection pool | DynamoDB: O(1) GetItem, TTL built-in |
| Sync click tracking in redirect | +3ms per redirect at 115K QPS compounds fast | Kinesis async: return 302 first |

### Real-World

| Service | Code | Redirect | Why |
|---|---|---|---|
| bit.ly | Counter Base58 | **302** | Analytics is core product |
| TinyURL | Hash MD5 prefix | 301 | No analytics needed |
| t.co | Counter | 301 | Twitter clients report engagement separately |

### Well-Architected Checklist

- [ ] **SEC 5** — Lambda in VPC; ElastiCache + DynamoDB in private subnets
- [ ] **SEC 6** — WAF rate-based rule: 100 req/IP/min on CloudFront
- [ ] **SEC 8** — HTTPS via ACM on CloudFront
- [ ] **REL 9** — DynamoDB on-demand; ElastiCache Multi-AZ
- [ ] **PERF 7** — DynamoDB not SQL for single-key lookups
- [ ] **PERF 4** — Negative caching (5min TTL) for 404s
- [ ] **OPS 5** — CDK: Lambda + API GW + DynamoDB + ElastiCache + CloudFront
- [ ] **OPS 6** — CloudWatch: Lambda p99 > 50ms; cache miss > 30%
- [ ] **COST 4** — Lambda for spiky viral traffic; DynamoDB on-demand
